# 127 · Fine-tune SmolVLA (LeRobot) on the TableTop dataset (Colab GPU; LeRobot 0.4.x)

In [ ]:
!pip install -q "lerobot[smolvla]>=0.4,<0.5"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Upload data/lerobot_v3/tabletop_panda_v1 (created by S12 notebook 117) as a zip, or push it to the Hub from your machine:
#   ds.push_to_hub()  ->  then use --dataset.repo_id=<your_user>/tabletop_panda_v1 below and drop --dataset.root
from google.colab import files
up = files.upload()                       # select tabletop_panda_v1.zip
!unzip -q -o tabletop_panda_v1.zip -d /content/tabletop_panda_v1
!ls /content/tabletop_panda_v1/meta

In [ ]:
import json
info = json.load(open("/content/tabletop_panda_v1/meta/info.json"))
print(info["codebase_version"], info["total_episodes"], "episodes", list(info["features"]))

In [ ]:
!lerobot-train \n  --policy.path=lerobot/smolvla_base \n  --dataset.repo_id=local/tabletop_panda_v1 \n  --dataset.root=/content/tabletop_panda_v1 \n  --rename_map='{"observation.images.front": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \n  --batch_size=32 --steps=6000 --save_freq=2000 \n  --output_dir=/content/outputs/smolvla_tabletop --job_name=smolvla_tabletop \n  --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false

In [ ]:
import torch, numpy as np
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ckpt = "/content/outputs/smolvla_tabletop/checkpoints/last/pretrained_model"
policy = SmolVLAPolicy.from_pretrained(ckpt).to("cuda").eval()
ds = LeRobotDataset("local/tabletop_panda_v1", root="/content/tabletop_panda_v1")
print("policy expects:", list(policy.config.input_features), "->", list(policy.config.output_features), "| chunk", policy.config.chunk_size)

Closed-loop evaluation: download the checkpoint folder and run it in TableTop locally with `lerobot` installed (see S13 lecture for the observation mapping: front→camera1, wrist→camera2, 8-D state, 7-D action).